In [2]:
import polars as pl
import numpy as np
from scipy import stats
import altair as alt

In [3]:
df = pl.scan_parquet("data/data_bias.parquet")

In [4]:
def ci_lower(x, confidence = .95):
    x = x.to_numpy()
    n = len(x)

    mean = np.mean(x)
    se = stats.sem(x)
    ci = se * stats.t.pdf((1 + confidence) / 2, n-1)

    return mean - ci

def ci_upper(x, confidence = .95):
    x = x.to_numpy()
    n = len(x)

    mean = np.mean(x)
    se = stats.sem(x)
    ci = se * stats.t.pdf((1 + confidence) / 2, n-1)

    return mean + ci

In [5]:
df.with_columns(((pl.col("adv distance") - pl.col("adv distance").mean()) / pl.col("adv distance").std()).over(["dataset", "bias_type"]))

FileNotFoundError: No such file or directory (os error 2): data/data_bias.parquet

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'with_columns'


In [6]:
cols = ["train acc", "train f1", "test acc", "test f1", 
               "adv acc", "adv f1", "adv distance", "clusters"]

def agg_metrics(cols):
    res = []
    for col in cols:
        res.extend([
            pl.col(col).mean().alias(f"{col}_mean"),
            pl.col(col).std().alias(f"{col}_std"),
            pl.col(col).map_batches(ci_lower, return_dtype=pl.Float32, returns_scalar=True).alias(f"{col}_ci_lb"),
            pl.col(col).map_batches(ci_upper, return_dtype=pl.Float32, returns_scalar=True).alias(f"{col}_ci_ub"),
        ])
    return res

In [6]:
df_stats = df.select(pl.exclude("seed")).with_columns(
    (
        (pl.col("adv distance") - pl.col("adv distance").mean()) / pl.col("adv distance").std()
        ).over(["dataset", "bias_type"])
    ).group_by(
    # pl.col("distribution"),
    # pl.col("model"),
    # pl.col("seed"),
    pl.col("bias"),
    pl.col("dataset"),
    pl.col("bias_type"),
    
).agg(
    agg_metrics(cols)
).sort([
    # pl.col("distribution"),
    # pl.col("seed"),
    # pl.col("depth"),
    pl.col("bias")
])

In [7]:
df_stats.collect()

bias,dataset,bias_type,train acc_mean,train acc_std,train acc_ci_lb,train acc_ci_ub,train f1_mean,train f1_std,train f1_ci_lb,train f1_ci_ub,test acc_mean,test acc_std,test acc_ci_lb,test acc_ci_ub,test f1_mean,test f1_std,test f1_ci_lb,test f1_ci_ub,adv acc_mean,adv acc_std,adv acc_ci_lb,adv acc_ci_ub,adv f1_mean,adv f1_std,adv f1_ci_lb,adv f1_ci_ub,adv distance_mean,adv distance_std,adv distance_ci_lb,adv distance_ci_ub,clusters_mean,clusters_std,clusters_ci_lb,clusters_ci_ub
f64,str,str,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32,f64,f64,f32,f32
0.1,"""wine_feature5""","""oversampling""",1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.893301,0.068122,0.892327,0.894274,0.892157,0.069832,0.891159,0.893155,0.447484,0.153218,0.445293,0.449674,0.351402,0.16986,0.348973,0.35383,-0.038321,0.973084,-0.052232,-0.02441,1.61,0.621171,1.60112,1.61888
0.1,"""iris_feature2""","""undersampling""",0.981358,0.017956,0.981101,0.981615,0.98129,0.018033,0.981033,0.981548,0.932667,0.061563,0.931787,0.933547,0.930995,0.064133,0.930078,0.931912,0.673111,0.135884,0.671169,0.675054,0.592429,0.167351,0.590037,0.594821,0.048906,0.871072,0.036454,0.061359,1.936667,0.304902,1.932308,1.941025
0.1,"""wine_feature4""","""undersampling""",0.992176,0.010834,0.992022,0.992331,0.992192,0.010909,0.992036,0.992348,0.885218,0.074353,0.884155,0.886281,0.883971,0.075614,0.882891,0.885052,0.436503,0.170055,0.434072,0.438934,0.34317,0.181194,0.34058,0.34576,0.099891,1.009896,0.085454,0.114328,1.66,0.604664,1.651356,1.668644
0.1,"""wine_feature11""","""oversampling""",1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.893867,0.066753,0.892913,0.894821,0.892515,0.069071,0.891528,0.893503,0.448301,0.164922,0.445943,0.450658,0.353525,0.181352,0.350933,0.356118,0.004621,1.042959,-0.010288,0.019531,1.573333,0.604848,1.564687,1.58198
0.1,"""wine_feature9""","""oversampling""",1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.894194,0.065852,0.893252,0.895135,0.892957,0.067703,0.891989,0.893925,0.447407,0.162208,0.445089,0.449726,0.35207,0.179252,0.349508,0.354633,-0.013755,1.022783,-0.028376,0.000867,1.61,0.631847,1.600967,1.619033
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.9,"""Car Evaluation_feature2""","""oversampling""",0.921226,0.021328,0.920962,0.92149,0.829916,0.043035,0.829383,0.830449,0.85912,0.038978,0.858637,0.859603,0.7255,0.078259,0.724531,0.726469,0.562044,0.090852,0.560918,0.563169,0.276536,0.072087,0.275643,0.277429,0.058897,1.131496,0.044883,0.072911,9.62,2.860254,9.584575,9.655425
0.9,"""wine_feature3""","""oversampling""",1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.892538,0.068622,0.891557,0.893519,0.891846,0.070275,0.890842,0.892851,0.468431,0.176098,0.465914,0.470949,0.378219,0.18543,0.375568,0.38087,0.069723,1.042605,0.054818,0.084627,1.606667,0.599851,1.598091,1.615242
0.9,"""Car Evaluation_feature0""","""undersampling""",0.39587,0.330092,0.391619,0.40012,0.312531,0.200521,0.309949,0.315113,0.368323,0.312796,0.364295,0.372351,0.274982,0.179997,0.272665,0.2773,0.339939,0.201578,0.337343,0.342535,0.182062,0.084597,0.180972,0.183151,-0.119064,0.587948,-0.126635,-0.111493,9.024324,3.511028,8.979115,9.069534


In [ ]:
base = alt.Chart(
    df_stats.collect()
)

scale = alt.Scale(
    domain=["train acc_mean", "test acc_mean", "clusters_mean"], 
    range=["blue", "red", "purple"]
)


dash_scale = alt.Scale(
    domain=["train acc_mean", "test acc_mean", "clusters_mean"],
    range=[[2, 4], [1, 0], [8, 4]]  # [dash_length, gap_length]
)

# Chart for accuracy metrics (left y-axis)
acc_chart = base.mark_line(strokeWidth=2).transform_fold(
    fold=["train acc_mean", "test acc_mean"],
    as_=["variable", "value"]
).encode(
    x=alt.X("bias:Q").title("Bias"),
    y=alt.Y('value:Q').title("Accuracy").scale(zero=False),
    color=alt.Color('variable:N', scale=scale, title="Metric"),
    strokeDash=alt.StrokeDash('variable:N', scale=dash_scale)
)


# Chart for adversarial distance (right y-axis)
adv_chart = base.mark_line(strokeWidth=2).transform_fold(
    fold=["clusters_mean"],
    as_=["variable", "value"]
).encode(
    x=alt.X("bias:Q").title("Bias"),
    y=alt.Y('value:Q').title("Adversarial Distance").axis(orient='right').scale(zero=False),
    color=alt.Color('variable:N', scale=scale, title="Metric"),
    strokeDash=alt.StrokeDash('variable:N', scale=dash_scale)

)

ci_band0 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="bias:Q",
    y=alt.Y("train acc_ci_lb:Q").axis(title="Accuracy", orient="left"),
    y2=alt.Y2("train acc_ci_ub:Q"),
    color = alt.value("blue"),
)

ci_band1 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="bias:Q",
    y=alt.Y("test acc_ci_lb:Q").axis(title="Accuracy", orient="left"),
    y2=alt.Y2("test acc_ci_ub:Q"),
    color = alt.value("red")
)

ci_band2 = base.mark_area(
    opacity=0.3,
    interpolate="basis",
).encode(
    x="bias:Q",
    y=alt.Y("adv distance_ci_lb:Q").axis(orient="right"),
    y2=alt.Y2("adv distance_ci_ub:Q"),
    color = alt.value("purple")
)

alt.layer(acc_chart + ci_band0 + ci_band1, adv_chart + ci_band2).resolve_scale(
    y='independent'
).facet(
    row="dataset",
    column="bias_type"
).resolve_scale(
    x="independent",
    y="independent"
)


alt.FacetChart(...)